# Етап 1.2 — як влаштований відгук

1. **Автоматично, без логіну:** куди веде кнопка «Відгукнутися» на DOU і Happy Monday, які ATS трапляються найчастіше, як виглядають їхні форми.
2. **Інтерактивно, з логіном** (`RUN_INTERACTIVE = True`): кандидат входить у свій акаунт, ми відкриваємо форму відгуку і знімаємо її структуру.

**Жодна форма тут не відправляється.** Код тільки читає поля й робить скриншоти.

In [1]:
import sys, json, re
from pathlib import Path
import pandas as pd
import feedparser

sys.path.insert(0, str(Path.cwd()))
from recon_utils import *

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

RUN_INTERACTIVE = False   # True — тільки коли людина поруч і готова залогінитись
SCREENS = SAMPLES_DIR / "raw" / "screens"
SCREENS.mkdir(parents=True, exist_ok=True)

## 1. DOU: на сайті чи зовні?

Беремо вакансії з кількох категорій. Для зовнішніх відгуків іде редирект `dou.ua/goto/vacancy/?id=…`, який віддає URL сайту компанії без логіну.

In [2]:
CATEGORIES = ["Python", "Front End", "QA", "Java", "DevOps"]
dou_rows = []
for cat in CATEGORIES:
    feed = feedparser.parse(fetch(f"https://jobs.dou.ua/vacancies/feeds/?category={cat.replace(' ', '%20')}"))
    for e in feed.entries[:16]:
        url = e.link.split("?")[0]
        page = parse_dou_page(fetch(url))
        row = {"category": cat, "url": url, "title": page["title"], "company": page["company"], "apply_type": page["apply_type"]}
        if page["apply_type"] == "external" and page["apply_href"]:
            chain = resolve_redirects(page["apply_href"])
            row["target"] = chain[-1]
            row["ats"] = classify_ats(chain[-1])
        dou_rows.append(row)
dou_apply = pd.DataFrame(dou_rows)
print("Вакансій перевірено:", len(dou_apply))
dou_apply.apply_type.value_counts().to_frame("count")

Вакансій перевірено: 80


,count
apply_type,
on_site,47
external,33


In [3]:
ats_stats = dou_apply[dou_apply.apply_type == "external"].ats.value_counts().to_frame("count")
ats_stats["share"] = (ats_stats["count"] / ats_stats["count"].sum()).round(2)
ats_stats

,count,share
ats,,
company_site,11,0.33
peopleforce,10,0.30
ashby,3,0.09
greenhouse,2,0.06
smartrecruiters,2,0.06
workable,2,0.06
bamboohr,1,0.03
lever,1,0.03
telegram,1,0.03


In [4]:
dou_apply[dou_apply.ats == "company_site"][["company", "target"]].head(15)

,company,target
13,Ciklum,https://explore-jobs.ciklum.com/en/sites/ciklum-career/job/4889/?utm_medium=...
15,ORIL,https://oril.hurma.work/public-vacancies/155?source=MTAwMQ==
24,Ciklum,https://explore-jobs.ciklum.com/en/sites/ciklum-career/job/4757?keyword=4757...
33,United Tech,https://unitedtech.ai/vacancies/manual_qa_engineer_mobile?source=dou
37,Sii Ukraine,https://sii.pl/en/job-ads/id/38401/?sii_source=18&utm_source=dou.ua&utm_medi...
40,United Tech,https://unitedtech.ai/en/vacancies/manual_qa_engineer_web/source=dou
42,Quarks,https://quarks.tech/career/207875db-a77f-42e2-b692-638b2254fb6b?utm_source=a...
44,Sii Ukraine,https://sii.pl/en/job-ads/id/38075/?sii_source=18&utm_source=dou.ua&utm_medi...
53,SFox,https://sfoxmodel.recman.page/job/483860?path=api
63,Sii Ukraine,https://sii.pl/en/job-ads/id/38347/?sii_source=18&utm_source=dou.ua&utm_medi...


Короткі посилання (`grnh.se`) і трекінгові редиректи вже розгорнуті `resolve_redirects`. Записи `company_site` — це власні сторінки кар'єри компаній. Частина з них сама вбудовує ATS (iframe), це видно вже в браузері (розділ 3).

## 2. Djinni і Happy Monday

In [5]:
dj_feed = feedparser.parse(fetch("https://djinni.co/jobs/rss/?primary_keyword=Python"))
dj_rows = []
for e in dj_feed.entries[:15]:
    p = parse_djinni_page(fetch(e.link))
    dj_rows.append({"title": p["title"], "company": p["company"], "direct_apply": p["direct_apply"],
                    "anon_apply_form": any(f["name"] == "anon_apply" for f in p["apply_form_fields"])})
dj_apply = pd.DataFrame(dj_rows)
print("Djinni directApply:", dj_apply.direct_apply.value_counts().to_dict())
print("Анонімна форма є на сторінці:", dj_apply.anon_apply_form.value_counts().to_dict())
dj_apply.head()

Djinni directApply: {True: 15}
Анонімна форма є на сторінці: {True: 15}


,title,company,direct_apply,anon_apply_form
0,Python Developer,PLANEKS,True,True
1,Senior Python Backend Engineer,Verbit,True,True
2,Full-Stack (Python+React) Engineer,Temy,True,True
3,Staff Backend Engineer (Python) / LUN App,ЛУН,True,True
4,Middle Python Developer,Ajax Systems,True,True


In [6]:
sm = fetch("https://happymonday.ua/job-sitemap.xml", ttl=3600)
hm_urls = re.findall(r"<loc>([^<]+)</loc>", sm)
hm_apply = pd.DataFrame([parse_hm_page(fetch(u)) | {"url": u} for u in hm_urls[:20]])
print("Відгук вимагає логіну:", hm_apply.apply_requires_login.value_counts().to_dict())
print("Вакансії з партнерських джерел:", hm_apply.partner_source.value_counts().to_dict())
hm_apply[["title", "company", "partner_source", "apply_requires_login"]].head(10)

Відгук вимагає логіну: {True: 20}
Вакансії з партнерських джерел: {True: 19, False: 1}


,title,company,partner_source,apply_requires_login
0,Executive Operations Coordinator at MacPaw,MacPaw,False,True
1,Android Developer (AI Remodel),Kiss My Apps,True,True
2,Full Stack Developer (React/Next.js + Node.js),Kiss My Apps,True,True
3,HR Business Partner,Kiss My Apps,True,True
4,Enterprise Account Manager,Retell AI,True,True
5,Procurement Manager,Retell AI,True,True
6,Deployment Strategist,Retell AI,True,True
7,Senior Affiliate Manager (Full Remote - Ukraine),EverAI,True,True
8,Growth Marketing Lead (B2B SaaS & Payments),Ruby Labs,True,True
9,Middle Full-Stack Developer (Next.js),Ruby Labs,True,True


## 3. Форми зовнішніх ATS (без логіну)

Для кожного ATS із розділу 1 відкриваємо одну вакансію в headless Chromium і знімаємо схему форми: поля, підписи, обов'язковість, поле для CV, наявність капчі. Якщо форма прихована за кнопкою «Apply», натискаємо її. **«Submit» не натискаємо ніколи.**

In [7]:
from playwright.async_api import async_playwright

APPLY_BTN = re.compile(r"^(apply|apply now|apply for this job|подати|відгукнутися|податися)", re.I)

async def form_schema(url, shot_name):
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page(locale="en-US")
        await page.goto(url, wait_until="domcontentloaded", timeout=45000)
        await page.wait_for_timeout(2500)
        schema = await page.evaluate(FORM_SCHEMA_JS)
        if len(schema["fields"]) < 2:
            btn = page.get_by_role("link", name=APPLY_BTN).or_(page.get_by_role("button", name=APPLY_BTN)).first
            if await btn.count():
                try:
                    await btn.click(timeout=8000)
                    await page.wait_for_timeout(3000)
                    schema = await page.evaluate(FORM_SCHEMA_JS)
                except Exception as e:
                    schema["click_error"] = f"{type(e).__name__}: {str(e)[:120]}"
        # some career pages embed the ATS in an iframe
        if len(schema["fields"]) < 2:
            for fr in page.frames[1:]:
                try:
                    s = await fr.evaluate(FORM_SCHEMA_JS)
                    if len(s["fields"]) > len(schema["fields"]):
                        schema = s | {"iframe": fr.url}
                except Exception:
                    pass
        await page.screenshot(path=str(SCREENS / f"{shot_name}.png"), full_page=True)
        await browser.close()
        return schema

samples = (dou_apply.dropna(subset=["ats"]).groupby("ats").head(1)
           .query("ats not in ['email', 'telegram', 'linkedin']"))
ats_forms = {}
for _, r in samples.iterrows():
    try:
        ats_forms[r.ats] = await form_schema(r.target, f"ats_{r.ats}")
    except Exception as e:
        ats_forms[r.ats] = {"error": f"{type(e).__name__}: {e}"[:200], "url": r.target}
pd.DataFrame([{"ats": k, "fields": len(v.get("fields", [])), "file_input": any(f["type"] == "file" for f in v.get("fields", [])),
               "required": sum(f["required"] for f in v.get("fields", [])), "captcha": v.get("captcha"),
               "url": (v.get("url") or "")[:70], "error": v.get("error") or v.get("click_error")} for k, v in ats_forms.items()])

,ats,fields,file_input,required,captcha,url,error
0,bamboohr,28,True,10,True,https://suntechinnovation.bamboohr.com/careers/247?source=aWQ9MTM%3D,NaN
1,greenhouse,14,True,12,True,https://job-boards.eu.greenhouse.io/altamirafreelance/jobs/5008541101?,NaN
2,smartrecruiters,0,False,0,False,https://jobs.smartrecruiters.com/SigmaSoftware2/744000155787789-full-s,NaN
3,lever,8,True,4,True,https://jobs.lever.co/ajax/5900fe25-1830-4e60-b880-96ab93bac2b9/apply,NaN
4,company_site,2,False,1,False,https://explore-jobs.ciklum.com/en/sites/ciklum-career/job/4889/apply/,NaN
5,peopleforce,9,True,0,True,https://iconic21.peopleforce.io/careers/v/245515-front-end-developer/a,NaN
6,workable,0,False,0,False,https://apply.workable.com/raiffeisen-ukraine/j/8989E3FB90,TimeoutError: Locator.click: Timeout 8000ms exceeded.\nCall log:\n - waitin...
7,ashby,19,True,6,True,https://jobs.ashbyhq.com/solidgate/a0a2b6af-3b98-40be-ba90-ecb25ccc756,NaN


In [8]:
# схема полів для кожного ATS
for ats, s in ats_forms.items():
    if s.get("fields"):
        print(f"\n=== {ats}  ({s['url'][:80]})")
        display(pd.DataFrame(s["fields"])[["label", "type", "required"]].head(25))


=== bamboohr  (https://suntechinnovation.bamboohr.com/careers/247?source=aWQ9MTM%3D)


,label,type,required
0,Please leave this field blank,text,False
1,First Name *,text,True
2,Last Name *,text,True
3,Email *,text,True
4,Phone *,text,True
5,file-input,file,True
6,"Website, Blog or Portfolio",text,False
7,LinkedIn URL,text,False
8,Yes,radio,False
9,No,radio,False



=== greenhouse  (https://job-boards.eu.greenhouse.io/altamirafreelance/jobs/5008541101?gh_src=7mm)


,label,type,required
0,First Name*,text,True
1,Last Name*,text,True
2,Email*,text,True
3,Country*,text,True
4,First Name*,text,True
5,Phone*,tel,True
6,Attach,file,False
7,Attach,file,False
8,Link to LinkedIn page*,textarea,True
9,Level of English*,text,True



=== lever  (https://jobs.lever.co/ajax/5900fe25-1830-4e60-b880-96ab93bac2b9/apply)


...

,label,type,required
0,Email Address,email,True
1,honeypot,text,False



=== peopleforce  (https://iconic21.peopleforce.io/careers/v/245515-front-end-developer/a/new?sourc)


,label,type,required
0,Full name,text,False
1,Email,text,False
2,,tel,False
3,Resume,file,False
4,Telegram username,text,False
5,Desired base compensation,text,False
6,,text,False
7,Location,text,False
8,locale,select-one,False



=== ashby  (https://jobs.ashbyhq.com/solidgate/a0a2b6af-3b98-40be-ba90-ecb25ccc756d/applicat)


,label,type,required
0,,file,False
1,Name,text,True
2,Email,email,True
3,Phone,tel,True
4,Resume,file,True
5,Location,text,False
6,Additional Files,file,False
7,Cover Letter,textarea,False
8,LinkedIn,url,False
9,"Job Boards (DOU, djinni, Glassdoor, Indeed, etc.)",radio,False


In [9]:
save_sample("apply_flows.json", {
    "dou": dou_apply.to_dict("records"), "djinni": dj_apply.to_dict("records"),
    "happymonday": hm_apply[["url", "title", "company", "partner_source", "apply_requires_login"]].to_dict("records"),
    "ats_forms": ats_forms,
})
print("Збережено samples/apply_flows.json")

Збережено samples/apply_flows.json


## 4. Інтерактивно: логін і форми на самих сайтах

Потрібна людина. Порядок:
1. Постав `RUN_INTERACTIVE = True` у першій клітинці.
2. Запусти `await login("djinni")`. Відкриється вікно Chromium: увійди в акаунт і закрий вікно. Сесія збережеться в `.browser_profiles/djinni/` (у `.gitignore`).
3. Запусти `await inspect_onsite_form("djinni", "<url вакансії>")`. Форма відкриється, ми знімемо її поля і скриншот. **Нічого не відправляється.**

Те саме для `dou` і `happymonday`.

In [10]:
PROFILES = Path.cwd().parent / ".browser_profiles"
LOGIN_URLS = {
    "djinni": "https://djinni.co/login",
    "dou": "https://dou.ua/login/",
    "happymonday": "https://happymonday.ua/profile/login",
}
# which element opens the apply form on each site
APPLY_SELECTORS = {
    "djinni": "button.js-inbox-toggle-reply-form",
    "dou": "a[href='#apply'], .reply a",
    "happymonday": "a.js_job_click",
}

async def login(site):
    assert RUN_INTERACTIVE, "Спершу постав RUN_INTERACTIVE = True"
    async with async_playwright() as p:
        ctx = await p.chromium.launch_persistent_context(str(PROFILES / site), headless=False, locale="uk-UA")
        page = ctx.pages[0] if ctx.pages else await ctx.new_page()
        await page.goto(LOGIN_URLS[site])
        print("Увійди в акаунт у відкритому вікні, потім закрий вікно браузера.")
        await page.wait_for_event("close", timeout=0)
        await ctx.close()
    print("Сесію збережено:", PROFILES / site)

async def inspect_onsite_form(site, url):
    assert RUN_INTERACTIVE, "Спершу постав RUN_INTERACTIVE = True"
    async with async_playwright() as p:
        ctx = await p.chromium.launch_persistent_context(str(PROFILES / site), headless=False, locale="uk-UA")
        page = ctx.pages[0] if ctx.pages else await ctx.new_page()
        await page.goto(url, wait_until="domcontentloaded")
        await page.wait_for_timeout(2000)
        btn = page.locator(APPLY_SELECTORS[site]).first
        if await btn.count():
            await btn.click()
            await page.wait_for_timeout(2500)
        schema = await page.evaluate(FORM_SCHEMA_JS)
        await page.screenshot(path=str(SCREENS / f"onsite_{site}.png"), full_page=True)
        await ctx.close()
    save_sample(f"onsite_form_{site}.json", schema)
    return pd.DataFrame(schema["fields"])

In [11]:
# await login("djinni")

In [12]:
# await inspect_onsite_form("djinni", "https://djinni.co/jobs/<id>-<slug>/")